# 2. First COF build (Windows edition)

Language: **English** | [中文](../cn/02_first_cof_build.ipynb)

This notebook uses the shared `cofkit` environment to build a TAPB–TFB imine COF on the `hcb` topology. The runnable cell passes cofkit CLI arguments through Python's `subprocess` module; the direct Python API workflow below it remains commented.

## Tutorial series

1. [Environment setup](01_environment_setup.ipynb)
2. **First COF build** (this notebook)
3. [Topology, connectivity, and linkage examples](03_topologies_connectivities_and_linkages.ipynb)
4. [Pore analysis with Zeo++](04_zeopp_pore_analysis.ipynb)

Run [Notebook 1](01_environment_setup.ipynb) first. This notebook declares `Python (cofkit)` as its default kernel and also uses `conda run -n cofkit` explicitly.

## Build TAPB–TFB on `hcb`

TAPB and TFB each expose three reactive groups, giving a 3+3 connection combination. The `imine_bridge` template joins amines to aldehydes, while `--topology hcb` makes the topology choice explicit and reproducible.

The command writes `summary.json` and places exported CIFs below `out/tutorial_02_first_build/cifs/`, grouped by coarse validation class.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("Conda was not found. Run the environment setup cell first.")

command = [
    conda, "run", "-n", "cofkit", "cofkit",
    "build", "single-pair",
    "--template-id", "imine_bridge",
    "--first-smiles", "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
    "--second-smiles", "C1=C(C=C(C=C1C=O)C=O)C=O",
    "--first-id", "tapb",
    "--second-id", "tfb",
    "--first-motif-kind", "amine",
    "--second-motif-kind", "aldehyde",
    "--topology", "hcb",
    "--target-dimensionality", "2D",
    "--output-dir", "out/tutorial_02_first_build",
]
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python API equivalent (commented out).
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb",
#     "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine",
#     num_conformers=4,
# )
# tfb = build_rdkit_monomer(
#     "tfb",
#     "TFB",
#     "C1=C(C=C(C=C1C=O)C=O)C=O",
#     "aldehyde",
#     num_conformers=4,
# )
# generator = BatchStructureGenerator(
#     BatchGenerationConfig(
#         allowed_reactions=("imine_bridge",),
#         target_dimensionality="2D",
#         topology_ids=("hcb",),
#         rdkit_num_conformers=4,
#         max_workers=1,
#         write_cif=True,
#     )
# )
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb,
#     tfb,
#     pair_id="tapb__tfb",
#     out_dir=Path("out/tutorial_02_first_build_python/cifs"),
# )
# print("attempted:", attempted)
# for summary in summaries:
#     print(summary.topology_id, summary.status, summary.score, summary.cif_path)

## Inspect the build output

The CIF may appear under `valid`, `warning`, or `needs_optimization`; that directory is a coarse geometry classification, not a different chemistry. Notebook 4 searches all of these subdirectories rather than assuming one class.

In [ ]:
import json
from pathlib import Path

output_dir = Path("out/tutorial_02_first_build")
summary_path = output_dir / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("Run the first-build cell before inspecting its output.")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("--- summary.json ---")
print(json.dumps(summary, indent=2))
print("\n--- exported CIF files ---")
print(*(path.resolve() for path in sorted((output_dir / "cifs").rglob("*.cif"))), sep="\n")

In [ ]:
# Python equivalent (commented out): inspect the structured report and locate CIFs.
# import json
# from pathlib import Path
# output_dir = Path("out/tutorial_02_first_build")
# summary = json.loads((output_dir / "summary.json").read_text())
# print(summary["successful_structures"], summary["cifs_written"])
# print(*sorted((output_dir / "cifs").rglob("*.cif")), sep="\n")

## What to keep

- `summary.json` records the requested chemistry, detected motif counts, attempted structures, scores, validation flags, and CIF paths.
- Each CIF starts with a `# COFid:` comment that captures monomers, topology, and linkage.
- These are initial generated structures. A warning or `needs_optimization` result should be optimized and revalidated before simulation or publication.
- Keep `out/tutorial_02_first_build/` in place: Notebook 4 uses its CIF as the default Zeo++ input.

Continue with [Notebook 3](03_topologies_connectivities_and_linkages.ipynb) to compare a small set of other build choices.